# 02 — Predictive performance

Accuracy, average precision, NLL and Brier score for MC-dropout, BNN and the deep
ensemble across all three use cases.

The point of this figure is negative: the three methods reach comparable accuracy,
so later differences in their uncertainty scores cannot be explained by one model
simply predicting better.


In [ ]:
import os, sys, pathlib

# repo root, whether the notebook is run from notebooks/ or from the root
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

# released data: the summary bundle (all notebooks) and the raw per-pass predictions
# (notebooks 04 and 06). Set the variables before starting Jupyter to use other locations.
os.environ.setdefault('UQFIG_BUNDLE', str(ROOT / 'figure_data_bundle'))
os.environ.setdefault('UQFIG_PREDS', str(ROOT / 'cluster_preds'))
OUT = ROOT / 'out'; OUT.mkdir(exist_ok=True)

from uq_genomics.figures import style as S
S.apply_rc()
print('bundle :', os.environ['UQFIG_BUNDLE'])
print('preds  :', os.environ['UQFIG_PREDS'])
print('output :', OUT)

In [ ]:
from uq_genomics.figures import combine_figures as CF
CF.performance_ext(str(OUT))


In [ ]:
from IPython.display import IFrame, display
p = OUT / 'COMBINED_performance_ext_all_usecases.pdf'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=560)) if p.exists() else None


## Per-seed accuracy

Where the BNN fails to converge, its uncertainty rises everywhere, which would
flatter the sensitivity comparison in notebook 04. Worth checking before reading
any single-seed figure.


In [ ]:
import numpy as np
from uq_genomics.figures import config_paths as CP, data_access as DA
for task, ver in [('rbp', 17), ('gex', 19)]:
    spec = CP.VERSIONS[task]
    lab = DA.bundle_part(f'labels_{task}')
    M = DA.load_comparison(task)['mean']
    print(f'--- {task.upper()} imbalanced single-label ---')
    for s in spec['seeds']:
        y = np.asarray(lab[('imbal_single', s)]).argmax(1)
        row = f'  seed {s:>2d}'
        for m in CP.MODELS:
            k = f'v_{ver}_s_{s}_imbal_single_{m}'
            if k in M:
                row += f'  {CP.MODEL_LABEL[m]}={np.asarray(M[k]).argmax(1).__eq__(y).mean():.3f}'
        print(row)
